# Spark 4.1: Declarative Data Pipeline Workshop

## Using Spark Declarative Pipelines (SDP)

---

**Version:** Apache Spark 4.1.x  
**Approach:** Declarative (Spark Declarative Pipelines API)  
**Companion Notebook:** `spark40_imperative_pipeline.ipynb` (run side-by-side)

---

### Workshop Overview

This notebook teaches you how to build a **production-grade data pipeline** using the new **Spark Declarative Pipelines (SDP)** framework introduced in Apache Spark 4.1. You will:

1. Understand the business domain and data model (same as 4.0 notebook)
2. Build a medallion architecture using SDP decorators
3. Learn the declarative approach to defining transformations
4. Compare this approach with the imperative style in Spark 4.0

### How to Use This Workshop

**If running side-by-side with Spark 4.0 notebook:**
- Open both notebooks in JupyterLab (drag one tab to create split view)
- Execute cells in the same order in both notebooks
- Compare the code structure, verbosity, and approach
- Note the section numbers match between notebooks

### What is Spark Declarative Pipelines?

SDP is a framework donated by Databricks to Apache Spark that lets you **define** your pipeline transformations using decorators, rather than **executing** them imperatively. The framework handles:

- Automatic dependency resolution
- Execution ordering
- Checkpointing for streaming
- Validation without running (dry-run)

### Prerequisites

```bash
# Start the lakehouse stack
./lakehouse start all

# Generate and load test data
./lakehouse testdata generate --days 90
./lakehouse testdata load
```

---

## Section 1: The Business Domain

### What is a Ghost Kitchen?

A **ghost kitchen** (also called a virtual kitchen, cloud kitchen, or dark kitchen) is a professional food preparation facility that produces food exclusively for delivery. Unlike traditional restaurants, ghost kitchens have:

- **No dine-in customers** - delivery only
- **Multiple virtual brands** - one physical kitchen operates 5-10 different "restaurants"
- **Optimized for throughput** - designed for high-volume delivery orders
- **Data-driven operations** - every aspect is tracked and optimized

### Our Fictional Company: Casper's Kitchens

In this workshop, we're building a data pipeline for **Casper's Kitchens**, a ghost kitchen operator with:

| Metric | Value |
|--------|-------|
| Virtual Brands | 20 (Burger Republic, Wok This Way, Pizza Planet, etc.) |
| Markets | 4 cities (San Francisco, Silicon Valley, Seattle, Austin) |
| Menu Items | 160 items across 10 food categories |
| Daily Orders | ~835 per day (varies by hour and day of week) |
| Data Volume | ~14.5M order lifecycle events over 90 days |

### The Business Questions We're Solving

The data team at Casper's Kitchens needs to answer:

1. **Operations:** What are the average kitchen prep times by brand? Where are bottlenecks?
2. **Delivery:** What's the average delivery time by location? What's the P95?
3. **Revenue:** Which brands generate the most revenue? What's the average order value?
4. **Growth:** Which brands are growing vs declining? How does demand vary by hour?

### Why a Data Pipeline?

Raw operational data is **messy and hard to query**. We need to:

1. **Clean** - Remove nulls, fix data quality issues
2. **Enrich** - Parse nested JSON, add computed fields
3. **Aggregate** - Pre-compute metrics for fast dashboards
4. **Organize** - Structure data in layers for different use cases

---

## Section 2: The Data Model

### Order Lifecycle Events

Every order generates multiple events as it moves through the fulfillment process:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ order_created   │────▶│ kitchen_started │────▶│ kitchen_finished│
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                                        │
                                                        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│    delivered    │◀────│driver_picked_up │◀────│   order_ready   │
└─────────────────┘     └─────────────────┘     └─────────────────┘
        ▲                       │
        │               ┌───────┴───────┐
        └───────────────│  driver_ping  │ (every 60 seconds)
                        └───────────────┘
```

### Event Schema

Each event contains:

| Field | Type | Description |
|-------|------|-------------|
| `event_id` | string | Unique identifier for this event |
| `event_type` | string | One of: order_created, kitchen_started, etc. |
| `ts` | string | ISO timestamp of the event |
| `order_id` | string | Links all events for one order |
| `location_id` | int | Which city/kitchen processed this |
| `sequence` | int | Order of events (1, 2, 3...) |
| `body` | string | JSON with event-specific data |

### The `body` JSON Field

The `body` field contains different data depending on event type:

```json
// order_created event
{
  "brand_id": 5,
  "item_ids": "[12, 34, 56]",
  "total": 45.97,
  "lat": 37.7749,
  "lng": -122.4194
}

// driver_ping event
{
  "driver_id": "DRV-1234",
  "lat": 37.7812,
  "lng": -122.4098
}
```

### Dimension Tables

| Table | Records | Description |
|-------|---------|-------------|
| `dim_brands` | 20 | Virtual restaurant brands |
| `dim_categories` | 10 | Food categories (Pizza, Asian, etc.) |
| `dim_items` | 160 | Menu items with prices |
| `dim_locations` | 4 | Delivery markets/cities |

---

## Section 3: The Medallion Architecture

We'll organize our data using the **medallion architecture**, a data design pattern that organizes data into layers:

```
┌─────────────────────────────────────────────────────────────────────┐
│                         MEDALLION ARCHITECTURE                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌──────────┐      ┌──────────┐      ┌──────────┐                 │
│   │  BRONZE  │─────▶│  SILVER  │─────▶│   GOLD   │                 │
│   │  (Raw)   │      │ (Cleaned)│      │(Business)│                 │
│   └──────────┘      └──────────┘      └──────────┘                 │
│                                                                      │
│   • Raw ingestion    • Quality filters  • Pre-aggregated            │
│   • Schema-on-read   • Parsed JSON      • Business metrics          │
│   • Full fidelity    • Enriched fields  • Dashboard-ready           │
│                      • Joined dims                                   │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### Why Medallion?

| Layer | Purpose | Users |
|-------|---------|-------|
| **Bronze** | Preserve raw data exactly as received | Data engineers, debugging |
| **Silver** | Single source of truth, cleaned and enriched | Data scientists, analysts |
| **Gold** | Pre-computed metrics for specific use cases | Dashboards, executives |

### Our Tables

| Layer | Table | Description |
|-------|-------|-------------|
| Bronze | `bronze.dim_*` | Dimension tables (brands, items, locations, categories) |
| Bronze | `bronze.orders` | Raw order lifecycle events |
| Silver | `silver.orders_enriched` | Cleaned events with parsed JSON and time features |
| Silver | `silver.order_lifecycle` | One row per order with all timestamps and durations |
| Gold | `gold.hourly_metrics` | Orders and revenue by hour and location |
| Gold | `gold.delivery_performance` | Delivery time stats by date and location |
| Gold | `gold.brand_summary` | Brand-level performance metrics |

---

## Section 4: Environment Setup

Now let's set up our Spark environment. In the **declarative approach**, we:

1. Import the `pipelines` module as `dp`
2. Define our tables using decorators
3. Let the framework manage execution

**Compare with Spark 4.0:** In the imperative approach, we created a SparkSession and called functions explicitly. In SDP, `spark` is injected automatically by the framework.

### Important Note on This Notebook

In a **production SDP pipeline**, you would:
1. Define all `@dp.*` decorated functions in a `.py` file
2. Run with `spark-pipelines run --spec spark-pipeline.yml`

In this **notebook**, we'll:
1. Simulate the declarative pattern interactively
2. Show what the decorator syntax looks like
3. Execute the transformations to see results

This gives you the best of both worlds: understanding the declarative syntax while getting immediate feedback.

In [ ]:
# =============================================================================
# SECTION 4: ENVIRONMENT SETUP
# =============================================================================
#
# In the DECLARATIVE approach (Spark 4.1 SDP), you would typically:
#   1. Import the pipelines module: from pyspark import pipelines as dp
#   2. Define functions with @dp.materialized_view decorators
#   3. Run via CLI: spark-pipelines run
#
# The framework handles SparkSession creation, execution order, and writes.
#
# For this NOTEBOOK, we'll create a SparkSession so we can run interactively,
# but we'll show the decorator syntax to illustrate the declarative approach.
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

# In production SDP, you would NOT create SparkSession - the framework does it.
# We create it here for interactive notebook execution.
spark = SparkSession.builder \
    .appName("Spark41_Declarative_Pipeline") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")
print("\n✓ Environment ready")
print("\nNote: In production, you would run: spark-pipelines run --spec spark-pipeline.yml")

In [ ]:
# =============================================================================
# DECLARATIVE IMPORTS (What you'd use in production)
# =============================================================================
#
# In a production SDP pipeline file, you would import:
#
#   from pyspark import pipelines as dp
#
# Then use decorators like:
#   @dp.materialized_view(name="bronze.dim_brands")
#   @dp.table(name="bronze.orders_streaming")  # for streaming
#   @dp.temporary_view  # for intermediate results
#
# The 'spark' variable is INJECTED by the framework - you don't create it.
# =============================================================================

# Attempt to import SDP module (may not be available in all environments)
try:
    from pyspark import pipelines as dp
    SDP_AVAILABLE = True
    print("✓ Spark Declarative Pipelines module available")
except ImportError:
    SDP_AVAILABLE = False
    print("⚠ SDP module not available - will show decorator syntax but execute imperatively")
    print("  Install with: pip install 'pyspark[pipelines]'")

### Understanding the Decorator Pattern

In SDP, you **declare** tables instead of **executing** them:

```python
# DECLARATIVE (SDP) - Define what the table should be
@dp.materialized_view(name="bronze.dim_brands")
def dim_brands():
    return spark.read.parquet("/data/dimensions/brands.parquet")

# vs

# IMPERATIVE (Traditional) - Execute and write explicitly
def load_dim_brands():
    df = spark.read.parquet("/data/dimensions/brands.parquet")
    df.write.mode("overwrite").saveAsTable("iceberg.bronze.dim_brands")
    return df.count()

# Must call it explicitly
load_dim_brands()
```

**Key difference:** In SDP, you just **return a DataFrame**. The framework handles the write.

---

## Section 5: Bronze Layer - Raw Data Ingestion

The Bronze layer is our **landing zone**. We ingest data with minimal transformation.

### Declarative Approach Characteristics

In the declarative style, we:
- Define tables with `@dp.materialized_view` decorators
- Return a DataFrame from each function
- Let the framework handle execution order and writes

**Compare with Spark 4.0:** The imperative approach required explicit function calls, loops, and write statements.

In [ ]:
# =============================================================================
# SECTION 5.1: LOAD DIMENSION TABLES (DECLARATIVE STYLE)
# =============================================================================
#
# In DECLARATIVE style, each table is defined with a decorator.
# The function RETURNS a DataFrame - no explicit write needed.
#
# Here's what the production code would look like:
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCTION SDP SYNTAX (how you would write this in a .py file)
# -----------------------------------------------------------------------------
#
# @dp.materialized_view(name="bronze.dim_categories")
# def dim_categories():
#     """Food categories dimension table."""
#     return spark.read.parquet("/data/dimensions/categories.parquet")
#
# @dp.materialized_view(name="bronze.dim_brands")
# def dim_brands():
#     """Ghost kitchen brands dimension table."""
#     return spark.read.parquet("/data/dimensions/brands.parquet")
#
# @dp.materialized_view(name="bronze.dim_items")
# def dim_items():
#     """Menu items dimension table."""
#     return spark.read.parquet("/data/dimensions/items.parquet")
#
# @dp.materialized_view(name="bronze.dim_locations")
# def dim_locations():
#     """Delivery locations dimension table."""
#     return spark.read.parquet("/data/dimensions/locations.parquet")
#
# -----------------------------------------------------------------------------

# For notebook execution, we'll define and run these:
def dim_categories():
    """Food categories dimension table.
    
    DECLARATIVE: Just return the DataFrame.
    The @dp.materialized_view decorator would handle:
      - Table naming (bronze.dim_categories)
      - Write mode (overwrite)
      - Iceberg catalog registration
    """
    return spark.read.parquet("/data/dimensions/categories.parquet")

def dim_brands():
    """Ghost kitchen brands dimension table."""
    return spark.read.parquet("/data/dimensions/brands.parquet")

def dim_items():
    """Menu items dimension table."""
    return spark.read.parquet("/data/dimensions/items.parquet")

def dim_locations():
    """Delivery locations dimension table."""
    return spark.read.parquet("/data/dimensions/locations.parquet")

# Execute and write (in SDP, this would be automatic)
print("Loading dimension tables to Bronze layer...")
print("=" * 50)

dimension_functions = [
    ("dim_categories", dim_categories),
    ("dim_brands", dim_brands),
    ("dim_items", dim_items),
    ("dim_locations", dim_locations),
]

for name, func in dimension_functions:
    df = func()
    df.write.mode("overwrite").saveAsTable(f"iceberg.bronze.{name}")
    print(f"  ✓ iceberg.bronze.{name}: {df.count():,} rows")

print("\n✓ All dimension tables loaded")

In [ ]:
# =============================================================================
# SECTION 5.2: EXPLORE DIMENSION DATA
# =============================================================================

# Show the brands - these are our virtual restaurants
print("BRANDS (Virtual Restaurants)")
print("=" * 50)
spark.table("iceberg.bronze.dim_brands").show(truncate=False)

In [ ]:
# Show the locations - these are our delivery markets
print("LOCATIONS (Delivery Markets)")
print("=" * 50)
spark.table("iceberg.bronze.dim_locations").show(truncate=False)

In [ ]:
# =============================================================================
# SECTION 5.3: LOAD ORDER EVENTS (FACT TABLE)
# =============================================================================
#
# DECLARATIVE PATTERN:
#   - Function returns a DataFrame with the transformation applied
#   - No explicit write - the decorator handles it
#   - Dependencies are inferred from spark.table() calls
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCTION SDP SYNTAX
# -----------------------------------------------------------------------------
#
# @dp.materialized_view(name="bronze.orders")
# def orders_batch():
#     """Order lifecycle events from batch parquet source."""
#     df = spark.read.parquet("/data/events/orders_90d.parquet")
#     return df.withColumn(
#         "event_timestamp",
#         f.to_timestamp(f.regexp_replace("ts", "T", " "))
#     )
#
# -----------------------------------------------------------------------------

def orders_batch():
    """Order lifecycle events from batch parquet source.
    
    DECLARATIVE INSIGHT:
      Notice this function is PURE - it just describes the transformation.
      It doesn't have side effects (no write statements).
      
      This purity is what enables:
        - Automatic dependency resolution
        - Dry-run validation
        - Parallel execution of independent tables
    """
    # Read raw data
    df = spark.read.parquet("/data/events/orders_90d.parquet")
    
    # Parse timestamp - minimal transformation for bronze
    return df.withColumn(
        "event_timestamp",
        f.to_timestamp(f.regexp_replace("ts", "T", " "))
    )

# Execute
print("Loading order events to Bronze layer...")
print("=" * 50)

orders_df = orders_batch()
orders_df.write.mode("overwrite").saveAsTable("iceberg.bronze.orders")
print(f"  ✓ iceberg.bronze.orders: {orders_df.count():,} rows")
print("\n✓ Bronze layer complete")

In [ ]:
# =============================================================================
# SECTION 5.4: EXPLORE ORDER DATA
# =============================================================================

# Show the schema
print("ORDER EVENTS SCHEMA")
print("=" * 50)
spark.table("iceberg.bronze.orders").printSchema()

In [ ]:
# Show sample data
print("SAMPLE ORDER EVENTS")
print("=" * 50)
spark.table("iceberg.bronze.orders") \
    .select("event_id", "event_type", "order_id", "location_id", "event_timestamp", "body") \
    .show(10, truncate=40)

In [ ]:
# Count events by type
print("EVENT TYPE DISTRIBUTION")
print("=" * 50)
spark.table("iceberg.bronze.orders") \
    .groupBy("event_type") \
    .count() \
    .orderBy(f.desc("count")) \
    .show()

### Section 5 Summary: Bronze Layer

**What we did:**
- Loaded 4 dimension tables (brands, categories, items, locations)
- Loaded ~14.5M order lifecycle events
- Added a parsed timestamp column

**Declarative patterns shown:**
- Pure functions that return DataFrames
- No explicit write statements in the function body
- Table naming via decorator (in production)

**Compare with Spark 4.0:** The imperative version used explicit loops and write calls. In SDP, the framework would iterate through all decorated functions and execute them in dependency order automatically.

---

## Section 6: Silver Layer - Data Transformation

The Silver layer is where we **clean, enrich, and transform** our data.

### Declarative Dependencies

In SDP, dependencies are **inferred automatically** from `spark.table()` calls:

```python
@dp.materialized_view(name="silver.orders_enriched")
def orders_enriched():
    # These spark.table() calls CREATE dependencies:
    orders = spark.table("iceberg.bronze.orders")       # depends on bronze.orders
    locations = spark.table("iceberg.bronze.dim_locations")  # depends on dim_locations
    # ...
```

The framework builds a DAG (Directed Acyclic Graph) and executes tables in the correct order.

In [ ]:
# =============================================================================
# SECTION 6.1: TRANSFORM ORDERS TO SILVER - ORDERS_ENRICHED
# =============================================================================
#
# DECLARATIVE PATTERN:
#   - Function reads from other tables using spark.table()
#   - These reads CREATE DEPENDENCIES that the framework tracks
#   - Function returns the transformed DataFrame
#   - NO print statements for progress - framework provides that
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCTION SDP SYNTAX
# -----------------------------------------------------------------------------
#
# @dp.materialized_view(name="silver.orders_enriched")
# def orders_enriched():
#     """Orders with parsed JSON body, time features, and location join."""
#     orders = spark.table("iceberg.bronze.orders")
#     locations = spark.table("iceberg.bronze.dim_locations")
#     # ... transformation ...
#     return enriched_df
#
# -----------------------------------------------------------------------------

def orders_enriched():
    """Orders with parsed JSON body, time features, and location join.
    
    DECLARATIVE INSIGHT:
      Notice how this function is SELF-CONTAINED.
      It declares its inputs (spark.table calls) and output (return value).
      The framework can analyze this to build a dependency graph.
      
    DEPENDENCIES (inferred from spark.table calls):
      - iceberg.bronze.orders
      - iceberg.bronze.dim_locations
    """
    
    # =========================================================================
    # READ INPUTS - These create dependencies
    # =========================================================================
    orders = spark.table("iceberg.bronze.orders")
    locations = spark.table("iceberg.bronze.dim_locations")
    
    # =========================================================================
    # TRANSFORM - Same logic as imperative, but no side effects
    # =========================================================================
    
    # Step 1: Filter nulls
    cleaned = orders.filter(
        f.col("event_id").isNotNull() &
        f.col("order_id").isNotNull() &
        f.col("event_timestamp").isNotNull()
    )
    
    # Step 2: Parse JSON body
    body_schema = StructType([
        StructField("brand_id", IntegerType(), True),
        StructField("item_ids", StringType(), True),
        StructField("total", DoubleType(), True),
        StructField("lat", DoubleType(), True),
        StructField("lng", DoubleType(), True),
        StructField("driver_id", StringType(), True),
    ])
    
    enriched = cleaned.withColumn("body_parsed", f.from_json("body", body_schema))
    
    # Step 3: Extract fields
    enriched = enriched.select(
        "event_id", "event_type", "event_timestamp", "ts", "ts_seconds",
        "order_id", "location_id", "sequence", "body",
        f.col("body_parsed.brand_id").alias("brand_id"),
        f.col("body_parsed.total").alias("order_total"),
        f.col("body_parsed.lat").alias("latitude"),
        f.col("body_parsed.lng").alias("longitude"),
        f.col("body_parsed.driver_id").alias("driver_id"),
    )
    
    # Step 4: Add time features
    enriched = enriched.withColumns({
        "event_hour": f.hour("event_timestamp"),
        "event_day_of_week": f.dayofweek("event_timestamp"),
        "is_weekend": f.when(
            f.dayofweek("event_timestamp").isin(1, 7),
            True
        ).otherwise(False),
        "event_date": f.to_date("event_timestamp"),
    })
    
    # Step 5: Join with locations
    locations_lookup = locations.select(
        f.col("id").alias("location_id"),
        f.col("city").alias("city_name"),
    )
    
    enriched = enriched.join(
        f.broadcast(locations_lookup),
        on="location_id",
        how="left"
    )
    
    # =========================================================================
    # RETURN - No write! The framework handles that.
    # =========================================================================
    return enriched

# Execute (in SDP, this would be automatic)
print("Transforming orders to Silver layer...")
print("=" * 50)

enriched_df = orders_enriched()
enriched_df.write.mode("overwrite").saveAsTable("iceberg.silver.orders_enriched")
print(f"  ✓ iceberg.silver.orders_enriched: {enriched_df.count():,} rows")

In [ ]:
# =============================================================================
# SECTION 6.2: EXPLORE ENRICHED DATA
# =============================================================================

print("ENRICHED ORDERS SCHEMA")
print("=" * 50)
spark.table("iceberg.silver.orders_enriched").printSchema()

In [ ]:
print("SAMPLE ENRICHED ORDERS")
print("=" * 50)
spark.table("iceberg.silver.orders_enriched") \
    .select(
        "event_type", "order_id", "city_name", "brand_id",
        "order_total", "event_hour", "is_weekend"
    ) \
    .filter(f.col("event_type") == "order_created") \
    .show(10)

In [ ]:
# =============================================================================
# SECTION 6.3: CREATE ORDER LIFECYCLE TABLE
# =============================================================================
#
# DECLARATIVE DEPENDENCY:
#   This table depends on silver.orders_enriched.
#   The framework would automatically run orders_enriched BEFORE this.
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCTION SDP SYNTAX
# -----------------------------------------------------------------------------
#
# @dp.materialized_view(name="silver.order_lifecycle")
# def order_lifecycle():
#     """Pivoted view with one row per completed order and duration metrics."""
#     orders = spark.table("iceberg.silver.orders_enriched")  # DEPENDENCY!
#     # ... pivot and duration logic ...
#     return completed_df
#
# -----------------------------------------------------------------------------

def order_lifecycle():
    """Pivoted view with one row per completed order and duration metrics.
    
    DEPENDENCY: iceberg.silver.orders_enriched
      The framework would ensure orders_enriched runs before this.
    """
    
    # Read input - creates dependency on orders_enriched
    orders = spark.table("iceberg.silver.orders_enriched")
    
    # Pivot events to columns
    lifecycle = orders.groupBy(
        "order_id", "location_id", "city_name"
    ).pivot(
        "event_type",
        ["order_created", "kitchen_started", "kitchen_finished", "order_ready",
         "driver_arrived", "driver_picked_up", "delivered"]
    ).agg(
        f.min("event_timestamp").alias("ts")
    )
    
    # Rename columns
    lifecycle = lifecycle.select(
        "order_id", "location_id", "city_name",
        f.col("order_created").alias("created_at"),
        f.col("kitchen_started").alias("kitchen_started_at"),
        f.col("kitchen_finished").alias("kitchen_finished_at"),
        f.col("order_ready").alias("order_ready_at"),
        f.col("driver_arrived").alias("driver_arrived_at"),
        f.col("driver_picked_up").alias("pickup_at"),
        f.col("delivered").alias("delivered_at"),
    )
    
    # Calculate durations
    lifecycle = lifecycle.withColumns({
        "kitchen_duration_min": (
            f.unix_timestamp("kitchen_finished_at") -
            f.unix_timestamp("kitchen_started_at")
        ) / 60,
        "delivery_duration_min": (
            f.unix_timestamp("delivered_at") -
            f.unix_timestamp("pickup_at")
        ) / 60,
        "total_duration_min": (
            f.unix_timestamp("delivered_at") -
            f.unix_timestamp("created_at")
        ) / 60,
    })
    
    # Filter to completed orders
    return lifecycle.filter(f.col("delivered_at").isNotNull())

# Execute
print("\nCreating order lifecycle table...")
print("=" * 50)

lifecycle_df = order_lifecycle()
lifecycle_df.write.mode("overwrite").saveAsTable("iceberg.silver.order_lifecycle")
print(f"  ✓ iceberg.silver.order_lifecycle: {lifecycle_df.count():,} rows (completed orders)")
print("\n✓ Silver layer complete")

In [ ]:
# =============================================================================
# SECTION 6.4: EXPLORE LIFECYCLE DATA
# =============================================================================

print("SAMPLE ORDER LIFECYCLE")
print("=" * 50)
spark.table("iceberg.silver.order_lifecycle") \
    .select(
        "order_id", "city_name",
        f.round("kitchen_duration_min", 1).alias("kitchen_min"),
        f.round("delivery_duration_min", 1).alias("delivery_min"),
        f.round("total_duration_min", 1).alias("total_min")
    ) \
    .show(10)

### Section 6 Summary: Silver Layer

**What we did:**
- Cleaned data by filtering null values
- Parsed JSON body field to extract structured data
- Added time-based features
- Joined with location dimension
- Created pivoted lifecycle table with duration metrics

**Declarative patterns shown:**
- Dependencies declared via `spark.table()` calls
- Pure functions without side effects
- Return DataFrame instead of writing

**Compare with Spark 4.0:** The transformation logic is identical, but the structure is cleaner. In SDP, you don't need print statements for progress - the framework provides that. You also don't need to worry about execution order.

---

## Section 7: Gold Layer - Business Aggregations

The Gold layer contains **pre-aggregated metrics** for dashboards and reports.

### Declarative Aggregations

In SDP, aggregations are just more `@dp.materialized_view` decorated functions. The framework handles:

- Running them after their dependencies
- Parallel execution where possible
- Consistent checkpointing

In [ ]:
# =============================================================================
# SECTION 7.1: HOURLY METRICS
# =============================================================================
#
# DECLARATIVE PATTERN:
#   Gold tables depend on Silver tables.
#   The framework ensures the dependency graph is respected.
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCTION SDP SYNTAX
# -----------------------------------------------------------------------------
#
# @dp.materialized_view(name="gold.hourly_metrics")
# def hourly_metrics():
#     """Hourly order metrics by location."""
#     orders = spark.table("iceberg.silver.orders_enriched")  # DEPENDENCY!
#     return orders.filter(...).groupBy(...).agg(...)
#
# -----------------------------------------------------------------------------

def hourly_metrics():
    """Hourly order metrics by location.
    
    DEPENDENCY: iceberg.silver.orders_enriched
    """
    orders = spark.table("iceberg.silver.orders_enriched")
    
    return orders.filter(
        f.col("event_type") == "order_created"
    ).groupBy(
        "event_date", "event_hour", "location_id", "city_name"
    ).agg(
        f.count("order_id").alias("order_count"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("brand_id").alias("unique_brands"),
    )

# Execute
print("Creating Gold layer aggregations...")
print("=" * 50)

hourly_df = hourly_metrics()
hourly_df.write.mode("overwrite").saveAsTable("iceberg.gold.hourly_metrics")
print(f"  ✓ iceberg.gold.hourly_metrics: {hourly_df.count():,} rows")

In [ ]:
# =============================================================================
# SECTION 7.2: DELIVERY PERFORMANCE
# =============================================================================

def delivery_performance():
    """Delivery performance metrics by date and location.
    
    DEPENDENCY: iceberg.silver.order_lifecycle
    """
    lifecycle = spark.table("iceberg.silver.order_lifecycle")
    
    return lifecycle.groupBy(
        f.to_date("created_at").alias("order_date"),
        "location_id", "city_name"
    ).agg(
        f.count("order_id").alias("completed_orders"),
        f.avg("kitchen_duration_min").alias("avg_kitchen_time_min"),
        f.avg("delivery_duration_min").alias("avg_delivery_time_min"),
        f.avg("total_duration_min").alias("avg_total_time_min"),
        f.percentile_approx("total_duration_min", 0.5).alias("median_total_time_min"),
        f.percentile_approx("total_duration_min", 0.95).alias("p95_total_time_min"),
    )

delivery_df = delivery_performance()
delivery_df.write.mode("overwrite").saveAsTable("iceberg.gold.delivery_performance")
print(f"  ✓ iceberg.gold.delivery_performance: {delivery_df.count():,} rows")

In [ ]:
# =============================================================================
# SECTION 7.3: BRAND SUMMARY
# =============================================================================

def brand_summary():
    """Brand-level summary metrics.
    
    DEPENDENCIES:
      - iceberg.silver.orders_enriched
      - iceberg.bronze.dim_brands
    """
    orders = spark.table("iceberg.silver.orders_enriched")
    brands = spark.table("iceberg.bronze.dim_brands")
    
    brand_metrics = orders.filter(
        f.col("event_type") == "order_created"
    ).groupBy("brand_id").agg(
        f.count("order_id").alias("total_orders"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("location_id").alias("locations_served"),
        f.min("event_date").alias("first_order_date"),
        f.max("event_date").alias("last_order_date"),
    )
    
    return brand_metrics.join(
        brands.select(f.col("id").alias("brand_id"), "name"),
        on="brand_id",
        how="left"
    ).select(
        "brand_id",
        f.col("name").alias("brand_name"),
        "total_orders", "total_revenue", "avg_order_value",
        "locations_served", "first_order_date", "last_order_date",
    )

brand_df = brand_summary()
brand_df.write.mode("overwrite").saveAsTable("iceberg.gold.brand_summary")
print(f"  ✓ iceberg.gold.brand_summary: {brand_df.count():,} rows")
print("\n✓ Gold layer complete")

In [ ]:
# =============================================================================
# SECTION 7.4: EXPLORE GOLD DATA
# =============================================================================

print("TOP PERFORMING BRANDS")
print("=" * 50)
spark.table("iceberg.gold.brand_summary") \
    .select(
        "brand_name",
        "total_orders",
        f.round("total_revenue", 2).alias("total_revenue"),
        f.round("avg_order_value", 2).alias("avg_order_value")
    ) \
    .orderBy(f.desc("total_revenue")) \
    .show(10)

In [ ]:
print("DELIVERY PERFORMANCE BY CITY")
print("=" * 50)
spark.table("iceberg.gold.delivery_performance") \
    .groupBy("city_name") \
    .agg(
        f.sum("completed_orders").alias("total_orders"),
        f.round(f.avg("avg_total_time_min"), 1).alias("avg_time_min"),
        f.round(f.avg("p95_total_time_min"), 1).alias("p95_time_min")
    ) \
    .orderBy(f.desc("total_orders")) \
    .show()

---

## Section 8: Pipeline Summary

### What We Built

```
DATA SOURCES                    BRONZE                 SILVER                    GOLD
─────────────────────────────────────────────────────────────────────────────────────────
                                                                                 
/data/dimensions/*.parquet ──▶ dim_brands                                       
                               dim_categories                                    
                               dim_items                                         
                               dim_locations ─────────┐                          
                                                      │                          
/data/events/orders.parquet ─▶ orders ───────────────┴─▶ orders_enriched ─────▶ hourly_metrics
                                                         │                       brand_summary
                                                         │                          
                                                         └─▶ order_lifecycle ──▶ delivery_performance
```

### Dependency Graph (What SDP Builds Automatically)

```
                    ┌────────────────┐
                    │  dim_locations │
                    └───────┬────────┘
                            │
┌─────────┐         ┌───────▼────────┐         ┌────────────────┐
│ orders  │────────▶│orders_enriched │────────▶│ hourly_metrics │
└─────────┘         └───────┬────────┘         └────────────────┘
                            │                          │
                            │                  ┌───────▼────────┐
                            │                  │ brand_summary  │◀── dim_brands
                            │                  └────────────────┘
                    ┌───────▼────────┐
                    │order_lifecycle │
                    └───────┬────────┘
                            │
                    ┌───────▼────────┐
                    │delivery_perf   │
                    └────────────────┘
```

In SDP, this graph is **inferred automatically** from `spark.table()` calls.

---

## Section 9: Declarative Approach - Tradeoffs

### Advantages of the Declarative Approach

| Advantage | Why It Matters |
|-----------|----------------|
| **Automatic dependency resolution** | No need to order function calls manually |
| **Automatic parallelism** | Independent tables run concurrently |
| **Less boilerplate** | No explicit write statements needed |
| **Built-in validation** | `dry-run` catches errors before execution |
| **CLI integration** | `spark-pipelines run` for production execution |
| **Consistent patterns** | All tables defined the same way |

### Disadvantages of the Declarative Approach

| Disadvantage | Impact |
|--------------|--------|
| **Framework dependency** | Requires Spark 4.1+ with SDP module |
| **Less control over execution** | Framework decides order and parallelism |
| **Debugging can be harder** | Errors may surface in framework, not your code |
| **Learning curve** | New concepts (decorators, DAG resolution) |
| **Less flexible for exploration** | Designed for production, not ad-hoc analysis |

### When to Use Declarative Style

The declarative approach is a good fit when:

- You have a well-defined pipeline with clear dependencies
- You want production-ready execution without custom orchestration
- You value consistency and reduced boilerplate
- You need automatic parallelism and checkpointing
- You're using Spark 4.1+ and can adopt the SDP framework

### Compare with Spark 4.0 Imperative

| Aspect | Imperative (4.0) | Declarative (4.1 SDP) |
|--------|------------------|-------------------|
| Execution order | Manual (you call functions) | Automatic (framework resolves) |
| Dependencies | Implicit (in your call order) | Explicit (from spark.table calls) |
| Boilerplate | More (read/write in each function) | Less (decorators + return) |
| Validation | Manual | Built-in dry-run |
| CLI integration | None | spark-pipelines command |
| Debugging | Easy (print anywhere) | Harder (framework abstraction) |
| Flexibility | High | Medium |
| Portability | Any Spark version | Spark 4.1+ only |

---

## Section 10: Running in Production

In production, you would:

### 1. Create a pipeline file (`pipeline_sdp.py`)

```python
from pyspark import pipelines as dp
from pyspark.sql import functions as f

@dp.materialized_view(name="bronze.orders")
def orders_batch():
    df = spark.read.parquet("/data/events/orders_90d.parquet")
    return df.withColumn("event_timestamp", ...)

@dp.materialized_view(name="silver.orders_enriched")
def orders_enriched():
    orders = spark.table("iceberg.bronze.orders")
    locations = spark.table("iceberg.bronze.dim_locations")
    # ... transformation ...
    return enriched_df

# ... more decorated functions ...
```

### 2. Create a config file (`spark-pipeline.yml`)

```yaml
name: lakehouse_pipeline
libraries:
  - file: pipeline_sdp.py
catalog: iceberg
database: bronze
storage: /tmp/lakehouse-checkpoint
```

### 3. Validate and Run

```bash
# Validate without executing
spark-pipelines dry-run --spec spark-pipeline.yml

# Run the pipeline
spark-pipelines run --spec spark-pipeline.yml
```

The framework will:
1. Parse all decorated functions
2. Build the dependency graph
3. Execute in correct order
4. Report progress and results

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================

# spark.stop()

print("\n" + "=" * 50)
print("WORKSHOP COMPLETE")
print("=" * 50)
print("\nYou've built a complete data pipeline using the declarative approach.")
print("\nKey takeaways:")
print("  1. Declarative: Define WHAT, not HOW")
print("  2. Dependencies are inferred from spark.table() calls")
print("  3. Framework handles execution order and writes")
print("  4. Use dry-run to validate before execution")
print("\nNext steps:")
print("  1. Compare with spark40_imperative_pipeline.ipynb")
print("  2. Try running: spark-pipelines run --spec scripts/spark-pipeline.yml")
print("  3. Add new tables and see how dependencies are resolved")